# Data Access Audit Trail Notebook

Portable audit of *actual* data access using `SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY` —
column-level read/write history, not just who is *granted* access. This is the direct
complement to the security posture notebook's masking-coverage section: that one shows what's
protected on paper, this one shows what's actually being queried and whether protections are
applying in practice. Run top to bottom. It surfaces:

1. **Who accessed sensitive-looking columns**, and how often
2. **Most-accessed sensitive tables**, ranked, to prioritize review
3. **Users with a broad access footprint** — touching many distinct sensitive columns, a proxy for over-broad access or bulk export behavior
4. **Policy enforcement in practice** — via `POLICIES_REFERENCED`, whether masking/row access policies actually applied during real queries
5. **Access tied to unload activity** — sensitive data read in the same query that wrote to an external location
6. **Write activity on sensitive objects** — via `OBJECTS_MODIFIED`
7. **A rolled-up summary**

### Prerequisites
- A role with `IMPORTED PRIVILEGES` on `SNOWFLAKE` (or `ACCOUNTADMIN`).
- `ACCESS_HISTORY` has a finite retention window that varies by account (commonly around a
  year) — confirm it for this environment before treating an absence of results as "never
  accessed."
- `POLICIES_REFERENCED` (Section 4) is a newer column and may not exist in older Snowflake
  releases; if that cell errors, it's safe to skip.
- Nothing here is destructive. Every cell reads.


In [ ]:
-- ============================================================
-- PARAMETERS
-- ============================================================
SET lookback_days = 90;
SET sensitive_column_patterns = 'SSN,SOCIAL_SEC,PATIENT,MRN,DOB,BIRTH,DIAGNOS,EMAIL,PHONE,ADDRESS,INSURANCE,MEDICAL,HEALTH,GENDER,RACE,ZIP,CREDIT_CARD';
SET broad_access_column_threshold = 15;  -- flag users touching more than this many distinct sensitive columns

SELECT $lookback_days AS lookback_days, $sensitive_column_patterns AS sensitive_column_patterns,
       $broad_access_column_threshold AS broad_access_column_threshold;


## 1. Who accessed sensitive-looking columns

Flattens `BASE_OBJECTS_ACCESSED` (the underlying tables actually read, after resolving
through any views) down to individual columns, then filters to column names that look
sensitive. Same name-matching caveat as the other notebooks: a starting point to confirm with
data owners, not data classification.


In [ ]:
-- Column-level read events on sensitive-looking columns
WITH flattened AS (
    SELECT
        ah.query_id,
        ah.query_start_time,
        ah.user_name,
        obj.value:"objectName"::string AS object_name,
        obj.value:"objectDomain"::string AS object_domain,
        col.value:"columnName"::string AS column_name
    FROM snowflake.account_usage.access_history ah,
         LATERAL FLATTEN(input => ah.base_objects_accessed) obj,
         LATERAL FLATTEN(input => obj.value:"columns", outer => true) col
    WHERE ah.query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
)
SELECT
    user_name,
    object_name,
    column_name,
    COUNT(*)               AS access_count,
    MIN(query_start_time)  AS first_access,
    MAX(query_start_time)  AS last_access
FROM flattened
WHERE EXISTS (
        SELECT 1 FROM TABLE(SPLIT_TO_TABLE($sensitive_column_patterns, ',')) p
        WHERE column_name ILIKE '%' || TRIM(p.value) || '%'
      )
GROUP BY user_name, object_name, column_name
ORDER BY access_count DESC
LIMIT 500;


## 2. Most-accessed sensitive tables

Rolled up to the table level so you can see where attention concentrates, independent of
which specific column within it was touched.


In [ ]:
-- Sensitive-table access ranked by total read volume and distinct users
WITH flattened AS (
    SELECT
        ah.query_id,
        ah.query_start_time,
        ah.user_name,
        obj.value:"objectName"::string AS object_name,
        col.value:"columnName"::string AS column_name
    FROM snowflake.account_usage.access_history ah,
         LATERAL FLATTEN(input => ah.base_objects_accessed) obj,
         LATERAL FLATTEN(input => obj.value:"columns", outer => true) col
    WHERE ah.query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
      AND EXISTS (
            SELECT 1 FROM TABLE(SPLIT_TO_TABLE($sensitive_column_patterns, ',')) p
            WHERE col.value:"columnName"::string ILIKE '%' || TRIM(p.value) || '%'
          )
)
SELECT
    object_name,
    COUNT(*)                    AS access_events,
    COUNT(DISTINCT user_name)   AS distinct_users,
    COUNT(DISTINCT column_name) AS distinct_sensitive_columns,
    MAX(query_start_time)       AS last_access
FROM flattened
GROUP BY object_name
ORDER BY access_events DESC
LIMIT 100;


## 3. Users with a broad sensitive-data access footprint

A user routinely touching a wide range of distinct sensitive columns across many tables is
either in a role that genuinely needs that (support, data engineering on that domain) or is a
signal worth a direct conversation. Either way it's worth knowing who's on this list before
you're the one answering for it later.


In [ ]:
-- Users ranked by breadth of sensitive-column access
WITH flattened AS (
    SELECT
        ah.user_name,
        obj.value:"objectName"::string AS object_name,
        col.value:"columnName"::string AS column_name
    FROM snowflake.account_usage.access_history ah,
         LATERAL FLATTEN(input => ah.base_objects_accessed) obj,
         LATERAL FLATTEN(input => obj.value:"columns", outer => true) col
    WHERE ah.query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
      AND EXISTS (
            SELECT 1 FROM TABLE(SPLIT_TO_TABLE($sensitive_column_patterns, ',')) p
            WHERE col.value:"columnName"::string ILIKE '%' || TRIM(p.value) || '%'
          )
)
SELECT
    user_name,
    COUNT(DISTINCT object_name) AS distinct_sensitive_tables,
    COUNT(DISTINCT column_name) AS distinct_sensitive_columns,
    COUNT(*)                    AS total_access_events
FROM flattened
GROUP BY user_name
HAVING COUNT(DISTINCT column_name) > $broad_access_column_threshold
ORDER BY distinct_sensitive_columns DESC;


## 4. Policy enforcement in practice

`POLICIES_REFERENCED` records which masking/row access policies actually evaluated during a
given query — the closest thing to direct proof that protection is working, versus just being
defined somewhere. If this returns nothing at all across the whole lookback window, either no
policies are enforced anywhere they're queried, or this Snowflake release doesn't populate the
column (check the release notes for when it was introduced in this account's version).


In [ ]:
-- Queries where a masking or row access policy was actually applied, sensitive columns only
WITH flattened AS (
    SELECT
        ah.query_id,
        ah.query_start_time,
        ah.user_name,
        pol.value:"policyName"::string   AS policy_name,
        pol.value:"policyKind"::string   AS policy_kind,
        pol.value:"columnName"::string   AS column_name
    FROM snowflake.account_usage.access_history ah,
         LATERAL FLATTEN(input => ah.policies_referenced) pol
    WHERE ah.query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
)
SELECT
    policy_name,
    policy_kind,
    COUNT(*)                   AS times_applied,
    COUNT(DISTINCT user_name)  AS distinct_users_affected,
    MAX(query_start_time)      AS last_applied
FROM flattened
GROUP BY policy_name, policy_kind
ORDER BY times_applied DESC;


Cross-check this against the security posture notebook's Section 6 (defined masking
policies and unmasked-sensitive-column list). A policy that exists but never shows up here
across a 90-day window either protects a column nobody queries, or isn't actually attached
where you'd expect — worth confirming which.


## 5. Access tied to unload activity

Joins sensitive-column reads against the same `query_id` performing a `COPY INTO <location>`
in the same statement — the closest signal in `ACCESS_HISTORY` to "sensitive data read and
then exported in one step." A multi-step export (query into a table, unload that table later)
won't join cleanly here since it spans two `query_id`s; treat this as one detection method
among several, not exhaustive.


In [ ]:
-- Sensitive-column reads that occurred in the same query as a COPY INTO an external/internal location
WITH sensitive_reads AS (
    SELECT DISTINCT
        ah.query_id
    FROM snowflake.account_usage.access_history ah,
         LATERAL FLATTEN(input => ah.base_objects_accessed) obj,
         LATERAL FLATTEN(input => obj.value:"columns", outer => true) col
    WHERE ah.query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
      AND EXISTS (
            SELECT 1 FROM TABLE(SPLIT_TO_TABLE($sensitive_column_patterns, ',')) p
            WHERE col.value:"columnName"::string ILIKE '%' || TRIM(p.value) || '%'
          )
)
SELECT
    qh.query_id,
    qh.user_name,
    qh.start_time,
    LEFT(qh.query_text, 250) AS query_text_preview
FROM snowflake.account_usage.query_history qh
JOIN sensitive_reads sr ON qh.query_id = sr.query_id
WHERE qh.start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND (qh.query_text ILIKE 'COPY INTO @%' OR qh.query_text ILIKE 'COPY INTO ''%')
ORDER BY qh.start_time DESC;


## 6. Write activity on sensitive objects

`OBJECTS_MODIFIED` covers inserts/updates/deletes/merges — worth knowing who can and does
write to tables holding sensitive data, separate from who merely reads them.


In [ ]:
-- Write events on tables that also appear in the sensitive-table list from Section 2
WITH sensitive_tables AS (
    SELECT DISTINCT obj.value:"objectName"::string AS object_name
    FROM snowflake.account_usage.access_history ah,
         LATERAL FLATTEN(input => ah.base_objects_accessed) obj,
         LATERAL FLATTEN(input => obj.value:"columns", outer => true) col
    WHERE EXISTS (
            SELECT 1 FROM TABLE(SPLIT_TO_TABLE($sensitive_column_patterns, ',')) p
            WHERE col.value:"columnName"::string ILIKE '%' || TRIM(p.value) || '%'
          )
),
writes AS (
    SELECT
        ah.query_id,
        ah.query_start_time,
        ah.user_name,
        mod.value:"objectName"::string AS object_name
    FROM snowflake.account_usage.access_history ah,
         LATERAL FLATTEN(input => ah.objects_modified) mod
    WHERE ah.query_start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
)
SELECT
    w.object_name,
    w.user_name,
    COUNT(*)               AS write_events,
    MAX(w.query_start_time) AS last_write
FROM writes w
JOIN sensitive_tables st ON w.object_name = st.object_name
GROUP BY w.object_name, w.user_name
ORDER BY write_events DESC
LIMIT 200;


## 7. Rolled-up summary

| Area | Finding | Follow-up |
|---|---|---|
| Sensitive-column access volume/users | *Section 1-2* | Confirm each accessing user/role has a legitimate need |
| Broad-footprint users | *Section 3* | Direct conversation with each; confirm role scope is intentional |
| Policy enforcement evidence | *Section 4* | Cross-check against security posture notebook's masking coverage |
| Read+unload correlation | *Section 5* | Investigate any hits directly; confirm sanctioned export process |
| Write access on sensitive tables | *Section 6* | Confirm write access list matches documented data-entry/ETL processes |


### Notes, caveats, and next steps

- **This is retrospective, not real-time.** For anything found here that looks actively
  concerning rather than a policy gap, that's an incident-response conversation, not something
  to resolve by re-running this notebook.
- **Column-name matching will both over- and under-flag.** Treat every result as "worth a
  human look," not a finding in itself — especially before this feeds into any compliance
  reporting.
- **Re-run on a schedule.** Unlike the one-time structural audits, access patterns drift
  continuously — this notebook is the one most worth scheduling monthly or tying to an
  alerting pipeline once the first pass is reviewed.
